# The F-test in Python

In this lesson, we will perform both the full and partial F-tests in Python.

Recall the Amazon book data — a dataset of **n = 325 books** with measurements including:
- `aprice`: The price listed on Amazon (dollars)
- `lprice`: The book's list price (dollars)
- `weight`: The book's weight (ounces)
- `pages`: The number of pages in the book
- `height`: The book's height (inches)
- `width`: The book's width (inches)
- `thick`: The thickness of the book (inches)
- `cover`: Whether the book is a hardcover or paperback

We'll explore a model using `lprice`, `pages`, and `width` to predict `aprice`. For all tests in this lesson, let **α = 0.05**.

## Setup

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

pd.set_option('display.float_format', '{:.6f}'.format)

## Load Data

In [3]:
url = ("https://raw.githubusercontent.com/bzaharatos/"
       "-Statistical-Modeling-for-Data-Science-Applications/"
       "refs/heads/master/Modern%20Regression%20Analysis/Datasets/amazon.txt")

amazon = pd.read_csv(url, sep='\t')
print(amazon.columns.tolist())
amazon.head()

['Title', 'Author', 'List Price', 'Amazon Price', 'Hard/ Paper', 'NumPages', 'Publisher', 'Pub year', 'ISBN-10', 'Height', 'Width', 'Thick', 'Weight (oz)']


,Title,Author,List Price,Amazon Price,Hard/ Paper,NumPages,Publisher,Pub year,ISBN-10,Height,Width,Thick,Weight (oz)
0,"1,001 Facts that Will Scare the S#*t Out of Yo...",Cary McNeal,12.950000,5.180000,P,304.000000,Adams Media,2010.000000,1605506249,7.800000,5.500000,0.800000,11.200000
1,21: Bringing Down the House - Movie Tie-In: Th...,Ben Mezrich,15.000000,10.200000,P,273.000000,Free Press,2008.000000,1416564195,8.400000,5.500000,0.700000,7.200000
2,100 Best-Loved Poems (Dover Thrift Editions),Smith,1.500000,1.500000,P,96.000000,Dover Publications,1995.000000,486285537,8.300000,5.200000,0.300000,4.000000
3,1421: The Year China Discovered America,Gavin Menzies,15.990000,10.870000,P,672.000000,Harper Perennial,2008.000000,0061564893,8.800000,6.000000,1.600000,28.800000
4,1493: Uncovering the New World Columbus Created,Charles C. Mann,30.500000,16.770000,P,720.000000,Knopf,2011.000000,0307265722,8.000000,5.200000,1.400000,22.400000


In [4]:
# Build the dataframe with renamed columns
df = pd.DataFrame({
    'aprice': amazon['Amazon Price'],
    'lprice': pd.to_numeric(amazon['List Price'], errors='coerce'),
    'pages':  amazon['NumPages'],
    'width':  amazon['Width'],
    'weight': amazon['Weight (oz)'],
    'height': amazon['Height'],
    'thick':  amazon['Thick'],
    'cover':  amazon['Hard/ Paper']
})

df.head()

,aprice,lprice,pages,width,weight,height,thick,cover
0,5.180000,12.950000,304.000000,5.500000,11.200000,7.800000,0.800000,P
1,10.200000,15.000000,273.000000,5.500000,7.200000,8.400000,0.700000,P
2,1.500000,1.500000,96.000000,5.200000,4.000000,8.300000,0.300000,P
3,10.870000,15.990000,672.000000,6.000000,28.800000,8.800000,1.600000,P
4,16.770000,30.500000,720.000000,5.200000,22.400000,8.000000,1.400000,P


## Data Cleaning

We'll clean the data to match the version used in the t-tests lesson — imputing missing values with column means and removing row 205.

In [5]:
for col in ['weight', 'pages', 'height', 'width', 'thick']:
    df[col] = df[col].fillna(df[col].mean())

# Remove row 205 (0-indexed: row 204)
df = df.drop(index=204).reset_index(drop=True)

df.describe()

,aprice,lprice,pages,width,weight,height,thick
count,324.000000,324.000000,324.000000,324.000000,324.000000,324.000000,324.000000
mean,13.010154,18.579753,335.844800,5.583719,12.477135,8.160966,0.908049
std,12.444872,14.252829,161.733280,0.868320,6.558926,0.913541,0.368576
min,0.770000,1.500000,24.000000,4.100000,1.200000,5.100000,0.100000
25%,8.597500,13.950000,208.000000,5.200000,7.800000,7.900000,0.600000
50%,10.200000,15.000000,320.000000,5.400000,11.200000,8.100000,0.900000
75%,13.032500,19.950000,416.000000,5.900000,16.000000,8.500000,1.100000
max,139.950000,139.950000,896.000000,9.500000,35.200000,12.100000,2.100000


## Full F-test

The **full model** for us includes `lprice`, `pages`, and `width` as predictors, with `aprice` as the response.

In Python (like R), the full F-test comes for free from the model summary — it appears at the bottom of the output as the F-statistic, degrees of freedom, and p-value.

This F-test asks: **is any predictor at all necessary?** The null hypothesis is that none of the predictors are needed — just an intercept is sufficient.

> **Important:** We should always look at the full F-test *before* individual t-tests. If the full F-test is not significant (large p-value), we shouldn't include any predictors. Only if the p-value is small does it make sense to examine individual t-tests.

In [6]:
# Fit the full model: aprice ~ lprice + pages + width
lm_amazon = smf.ols('aprice ~ lprice + pages + width', data=df).fit()
print(lm_amazon.summary())

                            OLS Regression Results                            
Dep. Variable:                 aprice   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1064.
Date:                Mon, 15 Jun 2026   Prob (F-statistic):          4.65e-166
Time:                        17:18:41   Log-Likelihood:                -888.04
No. Observations:                 324   AIC:                             1784.
Df Residuals:                     320   BIC:                             1799.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8630      1.574      0.548      0.5

**Interpretation of the full F-test:**

The F-statistic is very large (≈ 1064) and the p-value is effectively zero (< 4.65e-166). This tells us that **at least one predictor is necessary** — something beyond a simple intercept is needed to explain variability in Amazon price.

Looking at the individual t-tests now:
- `lprice`: highly significant — clearly doing most of the predictive work
- `pages`: significant, but less so
- `width`: **not significant** (p = 0.285) — no evidence its parameter differs from zero

Since `lprice` dominates, we might ask: do `pages` and `width` add anything beyond `lprice` alone?

## Partial F-test: Reduced (`lprice` only) vs. Full Model

We now test whether `pages` and/or `width` are needed beyond `lprice`.

- **H₀ (reduced):** Yᵢ = β₀ + β_lprice (lprice) + εᵢ  
- **H₁ (full):** number of pages or width (or both) should be included

To run a partial F-test in Python, we fit the reduced model and then pass both models to `anova_lm()`. The ANOVA table gives us the residual degrees of freedom, residual sums of squares, F-statistic, and p-value.

In [7]:
# Fit reduced model: aprice ~ lprice only
lm_amazon_reduced = smf.ols('aprice ~ lprice', data=df).fit()

# Partial F-test: reduced vs full
partial_f_test = anova_lm(lm_amazon_reduced, lm_amazon)
print(partial_f_test)

    df_resid         ssr  df_diff    ss_diff         F   Pr(>F)
0 322.000000 4846.160047 0.000000        NaN       NaN      NaN
1 320.000000 4557.840639 2.000000 288.319408 10.121263 0.000055


**Interpretation:**

The p-value (≈ 5.5 × 10⁻⁵) is much smaller than α = 0.05. We **reject the null** — the reduced model with only `lprice` is insufficient. At least one of `pages` or `width` adds meaningful explanatory power.

Since we already know from the t-test that `width` is not significant, we'll add back only `pages`:

**Yᵢ = β₀ + β_lprice (lprice) + β_pages (pages) + εᵢ**

This would be our candidate final model for predicting Amazon price or understanding the relationship between predictors and response.

## Partial F-test vs. Individual t-test: Are They Consistent?

An F-test can also compare two models that differ by **only one predictor**. This is useful for checking that the individual t-test and F-test give consistent results when testing a single parameter.

We compare:
- **ω (reduced):** Yᵢ = β₀ + β_lprice (lprice) + β_pages (pages) + εᵢ  
- **Ω (full):** Yᵢ = β₀ + β_lprice (lprice) + β_pages (pages) + β_width (width) + εᵢ

Let's check whether the F-test p-value matches the t-test p-value for `width`.

In [8]:
# Fit reduced model 2: aprice ~ lprice + pages
lm_amazon_reduced2 = smf.ols('aprice ~ lprice + pages', data=df).fit()

# Partial F-test: reduced2 vs full
partial_f_test2 = anova_lm(lm_amazon_reduced2, lm_amazon)
print("Partial F-test (lprice+pages vs full):")
print(partial_f_test2)

print("\nFull model summary (for t-test p-value of width):")
print(lm_amazon.summary())

Partial F-test (lprice+pages vs full):
    df_resid         ssr  df_diff   ss_diff        F   Pr(>F)
0 321.000000 4574.153134 0.000000       NaN      NaN      NaN
1 320.000000 4557.840639 1.000000 16.312494 1.145279 0.285346

Full model summary (for t-test p-value of width):
                            OLS Regression Results                            
Dep. Variable:                 aprice   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1064.
Date:                Mon, 15 Jun 2026   Prob (F-statistic):          4.65e-166
Time:                        17:19:50   Log-Likelihood:                -888.04
No. Observations:                 324   AIC:                             1784.
Df Residuals:                     320   BIC:                             1799.
Df Model:                           3                                         
Covariance Ty

**Result:** The p-value from the partial F-test and the p-value from the individual t-test for `width` are the same (≈ 0.285). This is **not a coincidence**.

The t-distribution and F-distribution are closely related:

> If X ~ t(n), then X² ~ F₁,ₙ

When a partial F-test removes exactly one predictor, it is mathematically equivalent to the individual t-test for that predictor. It's reassuring to see both approaches give consistent results.